# DeepTruth - 2-Stage Training: Image Deepfake Detection

**Run this notebook on Google Colab Pro (A100 GPU recommended)**

## Training Pipeline
1. **Stage 1 - Supervised Warm-up** (frozen backbones): Train fusion + classification heads first.
2. **Stage 2 - Full Fine-tuning** (all layers): End-to-end fine-tune with low, layer-wise LR.

**Why no contrastive pre-training?**  
SupCon stage was removed — it caused model collapse on FaceForensics++ (model learned to predict
everything as fake, AUROC=0.9999 on test set but non-discriminative on real-world images).

## Key safeguards against collapse
- Per-class accuracy tracked every epoch (`real_acc` and `fake_acc` shown separately)
- Automatic collapse detection: training stops if either class accuracy drops below 10%
- `WeightedRandomSampler` keeps real/fake balance
- Optional extra real-world images from Drive (Cell 7)


In [ ]:
# Cell 1 - Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm open-clip-torch albumentations einops
!pip install -q torchmetrics transformers

import os, json, math, random, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchmetrics import AUROC, Accuracy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# Cell 2 — Unpack data
DRIVE_ROOT = '/content/drive/MyDrive/deeptruth'
DATA_ZIP = f'{DRIVE_ROOT}/processed_faces.zip'
DATA_DIR = '/content/processed_faces'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

data_ready = os.path.exists(os.path.join(DATA_DIR, "real")) or os.path.exists(os.path.join(DATA_DIR, "splits"))
if not data_ready:
    print("Unpacking data...")
    os.makedirs(DATA_DIR, exist_ok=True)
    !unzip -q "{DATA_ZIP}" -d "{DATA_DIR}"
    print("Done.")
else:
    print("Data already unpacked.")

# Discover splits
SPLITS_DIR = os.path.join(DATA_DIR, 'splits')
if not os.path.exists(SPLITS_DIR):
    # Fallback: auto-discover face_crops folder and build splits on the fly
    FACE_DIR = None
    for root, dirs, files in os.walk(DATA_DIR):
        if 'face_crops' in dirs or ('real' in dirs and 'fake' in dirs):
            FACE_DIR = root
            break
    print(f'Face crop root: {FACE_DIR}')
else:
    FACE_DIR = None
    print(f'Splits dir: {SPLITS_DIR}')

# List available split files
if SPLITS_DIR and os.path.exists(SPLITS_DIR):
    for f in sorted(os.listdir(SPLITS_DIR)):
        print(' ', f)

In [ ]:
# Cell 3 — Dataset class with heavy augmentation

FAKE_TYPE_MAP = {
    'real': 0,
    'Deepfakes': 1,
    'Face2Face': 2,
    'FaceSwap': 3,
    'NeuralTextures': 4,
    'DeepFakeDetection': 5,
    'FaceShifter': 6,
    'gan_generated': 7,
    'diffusion_generated': 8,
    'unknown_fake': 9,
}

def build_file_list(splits_dir, split='train', face_dir=None):
    """Load file paths and labels from splits JSON or by auto-discovery."""
    split_file = os.path.join(splits_dir, f'{split}.json') if splits_dir else None
    if split_file and os.path.exists(split_file):
        with open(split_file) as f:
            entries = json.load(f)
        return [(e['path'], int(e['label']), e.get('fake_type', 'unknown_fake')) for e in entries]
    
    # Auto-discovery fallback
    assert face_dir is not None, 'Provide splits_dir or face_dir'
    entries = []
    for label_name, label in [('real', 0), ('fake', 1)]:
        d = os.path.join(face_dir, label_name)
        if not os.path.isdir(d):
            continue
        # Walk recursively to handle subdirectories (real/ffhq/, fake/stylegan2/, etc.)
        for root, _, fnames in os.walk(d):
            subdir = os.path.basename(root)
            fake_type = subdir if label == 1 else 'real'
            for fname in fnames:
                if fname.lower().endswith(('.jpg', '.png', '.jpeg')):
                    entries.append((os.path.join(root, fname), label, fake_type))
    
    random.seed(42)
    random.shuffle(entries)
    n = len(entries)
    if split == 'train':
        return entries[:int(0.8 * n)]
    elif split == 'val':
        return entries[int(0.8 * n):int(0.9 * n)]
    else:
        return entries[int(0.9 * n):]


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_train_transform():
    return A.Compose([
        A.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=7),
        ], p=0.4),
        A.OneOf([
            A.ImageCompression(quality_lower=60, quality_upper=100),
            A.Downscale(scale_min=0.5, scale_max=0.9),
        ], p=0.4),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transform():
    return A.Compose([
        A.Resize(height=224, width=224),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


class DeepTruthDataset(Dataset):
    def __init__(self, entries, transform=None, use_sbi=False):
        """entries: list of (path, label, fake_type)"""
        self.entries = entries
        self.transform = transform
        self.use_sbi = use_sbi

    def __len__(self):
        return len(self.entries)

    def _sbi_blend(self, img_np):
        """Self-Blended Image: paste face region from same image with slight warp."""
        h, w = img_np.shape[:2]
        # Random affine-warped copy
        angle = random.uniform(-10, 10)
        scale = random.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, scale)
        warped = cv2.warpAffine(img_np, M, (w, h))
        # Elliptical mask centered on face
        mask = np.zeros((h, w), dtype=np.float32)
        cx, cy = w // 2, int(h * 0.45)
        axes = (int(w * random.uniform(0.25, 0.4)), int(h * random.uniform(0.35, 0.5)))
        cv2.ellipse(mask, (cx, cy), axes, 0, 0, 360, 1.0, -1)
        # Smooth mask
        mask = cv2.GaussianBlur(mask, (31, 31), 11)
        mask = mask[:, :, np.newaxis]
        blended = (warped * mask + img_np * (1 - mask)).astype(np.uint8)
        return blended

    def __getitem__(self, idx):
        path, label, fake_type = self.entries[idx]
        img = Image.open(path).convert('RGB')
        img_np = np.array(img)

        # Self-Blended Image augmentation on real images during training
        if self.use_sbi and label == 0 and random.random() < 0.3:
            try:
                import cv2
                img_np = self._sbi_blend(img_np)
                label = 1
                fake_type = 'sbi_blend'
            except Exception:
                pass

        if self.transform:
            augmented = self.transform(image=img_np)
            img_tensor = augmented['image'].float()
        else:
            img_tensor = T.ToTensor()(img)

        type_id = FAKE_TYPE_MAP.get(fake_type, FAKE_TYPE_MAP['unknown_fake'])
        return img_tensor, torch.tensor(label, dtype=torch.long), torch.tensor(type_id, dtype=torch.long)

print('Dataset class defined.')

In [ ]:
# Cell 4 — Build DataLoaders
BATCH_SIZE = 32  # Reduce to 16 if OOM
NUM_WORKERS = 4

train_entries = build_file_list(SPLITS_DIR if os.path.exists(SPLITS_DIR) else None, 'train', FACE_DIR)
val_entries   = build_file_list(SPLITS_DIR if os.path.exists(SPLITS_DIR) else None, 'val',   FACE_DIR)
test_entries  = build_file_list(SPLITS_DIR if os.path.exists(SPLITS_DIR) else None, 'test',  FACE_DIR)

print(f'Train: {len(train_entries):,} | Val: {len(val_entries):,} | Test: {len(test_entries):,}')

# Weighted sampler to handle class imbalance
train_labels = [e[1] for e in train_entries]
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_ds = DeepTruthDataset(train_entries, get_train_transform(), use_sbi=True)
val_ds   = DeepTruthDataset(val_entries,   get_val_transform())
test_ds  = DeepTruthDataset(test_entries,  get_val_transform())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print(f'Class distribution — Real: {class_counts[0]:,} | Fake: {class_counts[1]:,}')

In [ ]:
# Cell 5 - Load model architecture
# Copy model_arch.py from Drive and import it.

import sys

DRIVE_ROOT  = '/content/drive/MyDrive/deeptruth'
CONFIG_PATH = f'{DRIVE_ROOT}/deeptruth_v2_config.json'
INIT_CKPT   = f'{DRIVE_ROOT}/deeptruth_v2_init.pth'   # optional - OK if missing

# Copy model_arch.py from Drive to /content/ so we can import it
arch_src = f'{DRIVE_ROOT}/model_arch.py'
assert os.path.exists(arch_src), f'model_arch.py not found at {arch_src}'
!cp "{arch_src}" /content/model_arch.py
sys.path.insert(0, '/content')

from model_arch import DeepTruthHybridV2, DeepTruthLoss
print('model_arch.py imported successfully.')

# Load config if present (optional - used only for metadata)
cfg = {}
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as _f:
        cfg = json.load(_f)
    print('Config loaded:', json.dumps(cfg, indent=2))
else:
    print('No config file found (optional) - continuing without it.')


In [ ]:
# Cell 6 - Instantiate model

model = DeepTruthHybridV2(
    num_fake_types=len(FAKE_TYPE_MAP),
    dropout=0.3,
).to(DEVICE)

# NOTE: We intentionally do NOT load the old checkpoint here.
# The previous checkpoint was trained on FaceForensics++ and collapsed
# (always predicted fake). We train from fresh pretrained backbone weights.
print('Model initialised from pretrained backbone weights (fresh training run).')

criterion = DeepTruthLoss(type_weight=0.3, focal_gamma=2.0, label_smoothing=0.1)

total_params  = sum(p.numel() for p in model.parameters())
trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M')


In [ ]:
# Cell 7 - Optional: Load extra real-world images from Drive
# Drop your own real photos into DeepTruth/extra_real/ on Drive to improve generalisation.
# Folder structure: extra_real/*.jpg  OR  extra_real/real/*.jpg + extra_real/fake/*.jpg

EXTRA_REAL_DIR = f'{DRIVE_ROOT}/extra_real'
extra_real_entries = []

if os.path.exists(EXTRA_REAL_DIR):
    for root, _, fnames in os.walk(EXTRA_REAL_DIR):
        subdir = os.path.basename(root).lower()
        label = 1 if 'fake' in subdir else 0
        fake_type = 'unknown_fake' if label == 1 else 'real'
        for fname in sorted(fnames):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                extra_real_entries.append((os.path.join(root, fname), label, fake_type))
    n_real_extra = sum(1 for e in extra_real_entries if e[1] == 0)
    n_fake_extra = sum(1 for e in extra_real_entries if e[1] == 1)
    print(f'Extra images found: {len(extra_real_entries)} '
          f'(real={n_real_extra}, fake={n_fake_extra})')
else:
    print('No extra_real folder found on Drive - skipping (optional).')
    print(f'  To add real-world photos, create {EXTRA_REAL_DIR}/ and place JPEG/PNG files there.')

# Inject extra entries into the training split (if any were found)
if extra_real_entries:
    train_entries_aug = train_entries + extra_real_entries
    aug_labels = [e[1] for e in train_entries_aug]
    aug_counts = np.bincount(aug_labels)
    aug_weights = 1.0 / aug_counts
    aug_sample_w = [aug_weights[l] for l in aug_labels]
    sampler_aug = WeightedRandomSampler(aug_sample_w, num_samples=len(aug_sample_w), replacement=True)
    train_ds_aug = DeepTruthDataset(train_entries_aug, get_train_transform(), use_sbi=True)
    train_loader = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, sampler=sampler_aug,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    print(f'Train loader updated: {len(train_loader)} batches '
          f'(real={aug_counts[0]:,}, fake={aug_counts[1]:,})')
else:
    print('Using original train_loader.')


In [ ]:
# Cell 8 - Training utilities

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best = None
        self.should_stop = False

    def __call__(self, metric):
        if self.best is None:
            self.best = metric
        elif (self.mode == 'max' and metric > self.best + self.min_delta) or \
             (self.mode == 'min' and metric < self.best - self.min_delta):
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


def save_checkpoint(model, optimizer, scaler, epoch, metrics, path, extra=None):
    ckpt = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'metrics': metrics,
    }
    if extra:
        ckpt.update(extra)
    torch.save(ckpt, path)


def load_checkpoint(model, optimizer, scaler, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    if optimizer and 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    if scaler and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])
    return ckpt['epoch'], ckpt.get('metrics', {})


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model. Returns loss, AUROC, overall acc, real_acc, fake_acc.
    Tracking real_acc and fake_acc separately is the primary collapse detector:
    collapse looks like fake_acc~1.0 and real_acc~0.0 (or vice-versa).
    """
    model.eval()
    total_loss = 0
    all_probs, all_labels = [], []
    auroc_metric = AUROC(task='binary').to(device)
    acc_metric   = Accuracy(task='binary').to(device)

    for imgs, labels, type_ids in tqdm(loader, desc='Eval', leave=False):
        imgs, labels, type_ids = imgs.to(device), labels.to(device), type_ids.to(device)
        with autocast():
            out = model(imgs)
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        total_loss += loss.item()
        probs = torch.softmax(out['binary_logit'], dim=1)[:, 1]
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    all_probs  = torch.cat(all_probs)
    all_labels = torch.cat(all_labels)
    preds = (all_probs > 0.5).long()

    auroc = auroc_metric(all_probs.to(device), all_labels.to(device)).item()
    acc   = acc_metric(preds.to(device), all_labels.to(device)).item()

    # Per-class accuracy - key collapse detector
    real_mask = all_labels == 0
    fake_mask = all_labels == 1
    real_acc = (preds[real_mask] == 0).float().mean().item() if real_mask.any() else float('nan')
    fake_acc = (preds[fake_mask] == 1).float().mean().item() if fake_mask.any() else float('nan')

    return {'loss': total_loss / len(loader), 'auroc': auroc, 'acc': acc,
            'real_acc': real_acc, 'fake_acc': fake_acc}


print('Training utilities ready.')


In [ ]:
# Cell 9 - Sanity Check: verify data balance and initial model outputs
# Run this BEFORE training to confirm the model is not already collapsed.

import matplotlib.pyplot as plt

# 1. Class distribution
train_labels_check = [e[1] for e in train_entries]
n_real_chk = sum(1 for l in train_labels_check if l == 0)
n_fake_chk = sum(1 for l in train_labels_check if l == 1)
print(f'Training set: {n_real_chk:,} real  |  {n_fake_chk:,} fake  |  '
      f'ratio={n_fake_chk/max(n_real_chk,1):.2f}:1')

# 2. Check initial model output distribution
print('\nChecking initial model output distribution (want ~0.3-0.7 for both classes)...')
model.eval()
sample_probs_real, sample_probs_fake = [], []
checked = 0
with torch.no_grad():
    for imgs, labels, _ in val_loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = torch.softmax(out['binary_logit'], dim=1)[:, 1].cpu()
        for p, l in zip(probs, labels):
            (sample_probs_real if l == 0 else sample_probs_fake).append(p.item())
        checked += len(labels)
        if checked >= 256:
            break

mean_real = sum(sample_probs_real) / max(len(sample_probs_real), 1)
mean_fake = sum(sample_probs_fake) / max(len(sample_probs_fake), 1)
print(f'  Mean fake-prob for REAL images: {mean_real:.4f}')
print(f'  Mean fake-prob for FAKE images: {mean_fake:.4f}')

if mean_real > 0.85:
    print('  WARNING: model already biased toward fake for real images.')
    print('  This suggests the checkpoint is from a collapsed training run.')
    print('  Consider training from scratch (set INIT_CKPT to a nonexistent path).')
elif mean_fake < 0.2:
    print('  WARNING: model already biased toward real for fake images.')
else:
    print('  OK: initial outputs look reasonable. Proceed to Stage 1.')

# 3. Show sample augmented crops
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Sample training crops (row 0=real, row 1=fake)')
real_shown, fake_shown = 0, 0
_mean = np.array([0.485, 0.456, 0.406])
_std  = np.array([0.229, 0.224, 0.225])
for imgs, labels, _ in train_loader:
    for img, label in zip(imgs, labels):
        img_np = np.clip(img.permute(1, 2, 0).numpy() * _std + _mean, 0, 1)
        row = 0 if label == 0 else 1
        col = real_shown if label == 0 else fake_shown
        if (label == 0 and real_shown < 4) or (label == 1 and fake_shown < 4):
            axes[row, col].imshow(img_np)
            axes[row, col].set_title('Real' if label == 0 else 'Fake')
            axes[row, col].axis('off')
            if label == 0: real_shown += 1
            else:          fake_shown += 1
        if real_shown == 4 and fake_shown == 4:
            break
    if real_shown == 4 and fake_shown == 4:
        break
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/sample_crops.png', dpi=100)
plt.show()
print('Sanity check complete. Proceed to Stage 1 training.')


In [ ]:
# Cell 10 - Stage 1: Supervised Warm-up
# Goal: train fusion + classification heads with FROZEN backbones.
# Freezing first lets the heads converge before disturbing pretrained backbone weights.

STAGE1_EPOCHS    = 10
STAGE1_LR        = 5e-4
STAGE1_CKPT      = f'{CKPT_DIR}/stage1_warmup_best.pth'
STAGE1_LAST_CKPT = f'{CKPT_DIR}/stage1_warmup_last.pth'

# Explicitly freeze all backbone streams; only train fusion + heads
for name, param in model.named_parameters():
    if any(k in name for k in ['clip_stream', 'effnet_stream', 'freq_stream', 'srm_stream', 'gram_stream']):
        param.requires_grad = False
    else:
        param.requires_grad = True

params_s1   = list(filter(lambda p: p.requires_grad, model.parameters()))
trainable_s1 = sum(p.numel() for p in params_s1)
print(f'Stage 1 trainable params: {trainable_s1/1e6:.2f}M (backbones frozen)')

opt1   = torch.optim.AdamW(params_s1, lr=STAGE1_LR, weight_decay=1e-4)
sched1 = torch.optim.lr_scheduler.OneCycleLR(
    opt1, max_lr=STAGE1_LR,
    steps_per_epoch=len(train_loader),
    epochs=STAGE1_EPOCHS,
    pct_start=0.1,
)
scaler1  = GradScaler()
early_s1 = EarlyStopping(patience=4, min_delta=0.002, mode='max')

best_auroc_s1 = 0
history_s1    = []

for epoch in range(STAGE1_EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_loader, desc=f'Stage1 E{epoch+1}/{STAGE1_EPOCHS}')

    for imgs, labels, type_ids in pbar:
        imgs, labels, type_ids = imgs.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt1.zero_grad()
        with autocast():
            out = model(imgs)
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        scaler1.scale(loss).backward()
        scaler1.unscale_(opt1)
        torch.nn.utils.clip_grad_norm_(params_s1, 1.0)
        scaler1.step(opt1)
        scaler1.update()
        sched1.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    avg_loss = running_loss / len(train_loader)
    history_s1.append({'train_loss': avg_loss, **val_metrics})
    print(f'  Stage1 E{epoch+1}: loss={avg_loss:.4f} '
          f'auroc={val_metrics["auroc"]:.4f} acc={val_metrics["acc"]:.4f} '
          f'real_acc={val_metrics["real_acc"]:.4f} fake_acc={val_metrics["fake_acc"]:.4f}')

    # Collapse guard
    if val_metrics['real_acc'] < 0.1 or val_metrics['fake_acc'] < 0.1:
        print('  COLLAPSE DETECTED (a class accuracy < 10%). Stopping.')
        print('  Reduce STAGE1_LR or check data balance and restart.')
        break

    if val_metrics['auroc'] > best_auroc_s1:
        best_auroc_s1 = val_metrics['auroc']
        save_checkpoint(model, opt1, scaler1, epoch, val_metrics, STAGE1_CKPT)
        print(f'    -> Best AUROC: {best_auroc_s1:.4f} (saved)')

    if early_s1(val_metrics['auroc']):
        print('Early stopping triggered.')
        break

save_checkpoint(model, opt1, scaler1, epoch, val_metrics, STAGE1_LAST_CKPT)
print(f'Stage 1 complete. Best val AUROC: {best_auroc_s1:.4f}')


In [ ]:
# Cell 11 - Stage 2: Full Fine-tuning
# Goal: unfreeze all layers, fine-tune end-to-end with low LR and layer-wise LR decay

STAGE2B_EPOCHS    = 15
STAGE2B_BASE_LR   = 3e-5
STAGE2B_CKPT      = f'{CKPT_DIR}/stage2_finetune_best.pth'
STAGE2B_LAST_CKPT = f'{CKPT_DIR}/stage2_finetune_last.pth'

# Load best Stage 1 weights
ckpt = torch.load(STAGE1_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded Stage 1 best checkpoint (val AUROC: {ckpt["metrics"]["auroc"]:.4f})')

# Unfreeze ALL parameters
for param in model.parameters():
    param.requires_grad = True

# Layer-wise LR: backbones get 10x lower LR than heads
backbone_params, head_params = [], []
for name, param in model.named_parameters():
    if any(k in name for k in ['clip_stream', 'effnet_stream', 'freq_stream', 'srm_stream', 'gram_stream']):
        backbone_params.append(param)
    else:
        head_params.append(param)

opt3 = torch.optim.AdamW([
    {'params': backbone_params, 'lr': STAGE2B_BASE_LR * 0.1},   # 3e-6 for backbones
    {'params': head_params,     'lr': STAGE2B_BASE_LR},          # 3e-5 for heads
], weight_decay=1e-4)

sched3 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt3, T_0=5, T_mult=2, eta_min=1e-7
)
scaler3  = GradScaler()
early_s3 = EarlyStopping(patience=6, min_delta=0.001, mode='max')

best_auroc_s3 = 0
history_s2b   = []

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Stage 2 trainable params: {total_params/1e6:.1f}M (all unfrozen)')

for epoch in range(STAGE2B_EPOCHS):
    model.train()
    running_loss = 0
    pbar = tqdm(train_loader, desc=f'Stage2 E{epoch+1}/{STAGE2B_EPOCHS}')

    for imgs, labels, type_ids in pbar:
        imgs, labels, type_ids = imgs.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt3.zero_grad()
        with autocast():
            out = model(imgs)
            loss = criterion(out['binary_logit'], labels, out['type_logit'], type_ids)
        scaler3.scale(loss).backward()
        scaler3.unscale_(opt3)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler3.step(opt3)
        scaler3.update()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    sched3.step()
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    avg_loss = running_loss / len(train_loader)
    history_s2b.append({'train_loss': avg_loss, **val_metrics})
    lrs = [g['lr'] for g in opt3.param_groups]
    print(f'  Stage2 E{epoch+1}: loss={avg_loss:.4f} '
          f'auroc={val_metrics["auroc"]:.4f} acc={val_metrics["acc"]:.4f} '
          f'real_acc={val_metrics["real_acc"]:.4f} fake_acc={val_metrics["fake_acc"]:.4f} '
          f'lrs={[f"{l:.2e}" for l in lrs]}')

    # Collapse guard
    if val_metrics['real_acc'] < 0.1 or val_metrics['fake_acc'] < 0.1:
        print('  COLLAPSE DETECTED (a class accuracy < 10%). Stopping.')
        break

    if val_metrics['auroc'] > best_auroc_s3:
        best_auroc_s3 = val_metrics['auroc']
        save_checkpoint(model, opt3, scaler3, epoch, val_metrics, STAGE2B_CKPT)
        print(f'    -> Best AUROC: {best_auroc_s3:.4f} (saved)')

    if early_s3(val_metrics['auroc']):
        print('Early stopping triggered.')
        break

save_checkpoint(model, opt3, scaler3, epoch, val_metrics, STAGE2B_LAST_CKPT)
print(f'Stage 2 complete. Best val AUROC: {best_auroc_s3:.4f}')


In [ ]:
# Cell 12 — Final evaluation on test set

# Load best checkpoint
ckpt = torch.load(STAGE2B_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]+1})')

test_metrics = evaluate(model, test_loader, criterion, DEVICE)
print('\n=== Test Set Results ===')
print(f'Loss:  {test_metrics["loss"]:.4f}')
print(f'AUROC: {test_metrics["auroc"]:.4f}')
print(f'Acc:   {test_metrics["acc"]:.4f}')

# Detailed confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels, _ in tqdm(test_loader, desc='Collecting predictions'):
        imgs = imgs.to(DEVICE)
        with autocast():
            out = model(imgs)
        probs = torch.softmax(out['binary_logit'], dim=1)[:, 1]
        preds = (probs > 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(labels.numpy())

print('\nClassification Report:')
print(classification_report(all_labels_list, all_preds, target_names=['Real', 'Fake']))

cm = confusion_matrix(all_labels_list, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Test AUROC={test_metrics["auroc"]:.3f})')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/confusion_matrix_image.png', dpi=150)
plt.show()

In [ ]:
# Cell 13 — Training history plots

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve (all stages)
history_s1_losses = history_s1
s2_losses = [h['train_loss'] for h in history_s1]
s3_losses = [h['train_loss'] for h in history_s2b]

all_losses = history_s1_losses + s2_losses + s3_losses
x = list(range(1, len(all_losses) + 1))
axes[0].plot(x, all_losses, 'b-', label='Train Loss')
axes[0].axvline(len(history_s1_losses), color='orange', linestyle='--', label='S1 end')
axes[0].axvline(len(history_s1_losses) + len(s2_losses), color='red', linestyle='--', label='S2 end')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (All Stages)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUROC curve (stages 2 and 3)
s2_auroc = [h['auroc'] for h in history_s1]
s3_auroc = [h['auroc'] for h in history_s2b]
all_auroc = s2_auroc + s3_auroc
x2 = list(range(1, len(all_auroc) + 1))
axes[1].plot(x2, all_auroc, 'g-', label='Val AUROC')
axes[1].axvline(len(s2_auroc), color='orange', linestyle='--', label='S2 end')
axes[1].axhline(test_metrics['auroc'], color='red', linestyle=':', label=f'Test AUROC={test_metrics["auroc"]:.3f}')
axes[1].set_xlabel('Supervised Epoch')
axes[1].set_ylabel('AUROC')
axes[1].set_title('Validation AUROC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('DeepTruth Image Model — Training History', fontsize=14)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_history_image.png', dpi=150)
plt.show()

# Save all history
full_history = {
    'stage1_supcon': history_s1,
    'stage2_warmup': history_s1,
    'stage3_finetune': history_s2b,
    'test_metrics': test_metrics,
}
with open(f'{CKPT_DIR}/image_training_history.json', 'w') as f:
    json.dump(full_history, f, indent=2)
print('History saved.')

In [ ]:
# Cell 14 — Grad-CAM explainability preview

class GradCAM:
    """Grad-CAM on the EfficientNet stream's last conv block."""
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        # Hook into EfficientNet last block
        target = model.effnet_stream.backbone.blocks[-1]
        target.register_forward_hook(self._save_activation)
        target.register_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(self, img_tensor, class_idx=1):
        """img_tensor: (1, 3, 224, 224)"""
        self.model.eval()
        img_tensor.requires_grad_(True)
        out = self.model(img_tensor)
        logit = out['binary_logit'][0, class_idx]
        self.model.zero_grad()
        logit.backward()

        grads  = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam    = (grads * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam    = F.relu(cam)
        cam    = cam - cam.min()
        cam    = cam / (cam.max() + 1e-8)
        cam_np = cam.squeeze().detach().cpu().numpy()
        return cam_np


import cv2 as _cv2

gradcam = GradCAM(model)

# Show 4 examples from test set
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
mean = np.array(IMAGENET_MEAN)
std  = np.array(IMAGENET_STD)

sample_indices = random.sample(range(len(test_ds)), 4)
for col, idx in enumerate(sample_indices):
    img_tensor, label, _ = test_ds[idx]
    inp = img_tensor.unsqueeze(0).to(DEVICE)

    # Original image (denormalize)
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * std + mean).clip(0, 1)

    # Grad-CAM
    cam = gradcam.generate(inp.clone(), class_idx=1)
    cam_resized = _cv2.resize(cam, (224, 224))
    heatmap = plt.cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * img_np + 0.5 * heatmap

    # Get prediction
    with torch.no_grad():
        out = model(inp)
        prob = torch.softmax(out['binary_logit'], dim=1)[0, 1].item()

    gt = 'Real' if label.item() == 0 else 'Fake'
    pred = 'Fake' if prob > 0.5 else 'Real'
    color = 'green' if gt == pred else 'red'

    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f'GT: {gt}', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f'Pred: {pred} ({prob:.2f})', fontsize=10, color=color)
    axes[1, col].axis('off')

plt.suptitle('Grad-CAM Explainability (Image Model)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/gradcam_examples.png', dpi=150)
plt.show()

In [ ]:
# Cell 15 - Save final image model to Drive

FINAL_IMAGE_MODEL = f'{CKPT_DIR}/deeptruth_image_model_final.pth'

# Load the best fine-tuned checkpoint
ckpt = torch.load(STAGE2B_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best Stage 2 checkpoint '
      f'(epoch {ckpt["epoch"]+1}, val AUROC={ckpt["metrics"]["auroc"]:.4f})')

# Save everything the backend needs for deployment
torch.save({
    'model_state':   model.state_dict(),
    'config':        cfg if os.path.exists(CONFIG_PATH) else {},
    'fake_type_map': FAKE_TYPE_MAP,
    'num_fake_types': len(FAKE_TYPE_MAP),
    'test_metrics':  test_metrics,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std':  IMAGENET_STD,
}, FINAL_IMAGE_MODEL)

size_mb = os.path.getsize(FINAL_IMAGE_MODEL) / 1e6
print(f'Final image model saved: {FINAL_IMAGE_MODEL}')
print(f'File size: {size_mb:.1f} MB')
print(f'Test AUROC: {test_metrics["auroc"]:.4f}')
print(f'Test Acc:   {test_metrics["acc"]:.4f}')

# Verify checkpoint key names match what the backend expects
state_keys = list(model.state_dict().keys())
print(f'\nFirst 5 checkpoint keys (must match backend model_arch.py):')
for k in state_keys[:5]:
    print(f'  {k}')
expected_prefixes = ['clip_stream', 'effnet_stream', 'freq_stream', 'srm_stream', 'gram_stream']
missing = [p for p in expected_prefixes if not any(k.startswith(p) for k in state_keys)]
if missing:
    print(f'WARNING: expected prefixes not found in checkpoint: {missing}')
    print('This means key names have changed - update backend/app/services/model_loader.py')
else:
    print('All expected key prefixes found. Checkpoint is compatible with backend.')

print('\nReady to deploy. Copy deeptruth_image_model_final.pth to models/ in the backend.')
